## The goal is to come up with a predictive model which helps prioritize quotes with highest chance of cross selling the product

  ### Case Details: 
  
  #### Existing customers have been quoted for "Product X" and the status of each quote is marked as Sold or Lost. The relavent data is stored in three files: 
  #### 1. `employers.csv` - holds basic customer data, such as number of years as a client, number of employees and Industry type. 
  #### 2. `geography.csv` - contains general geographic location by zip code, such as longitude, latitude, population etc... 
  #### 3. `quotes.csv` - contains the quote status (lost or sold) by the customer's ID. 
  #### 4. `dictionary.csv` - contains additional information on what each data set contains  

In [66]:
!pip install scikit-learn

In [67]:
# Import Packages

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pylab as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans


from scipy import stats
from scipy.stats import norm, skew




In [68]:
# Load Data
employer_data = pd.read_csv("data/employers.csv", encoding="cp1252")
geography_data = pd.read_csv("data/geography.csv", encoding="cp1252")
quotes_data = pd.read_csv("data/quotes.csv", encoding="cp1252")


In [69]:
employer_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 65356 entries, 0 to 65355
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   EmployerId       65356 non-null  int64
 1   ClientTenure     65356 non-null  int64
 2   Employees        65356 non-null  int64
 3   Industry         57125 non-null  str  
 4   NetworkStrength  65356 non-null  str  
 5   ZipCode          65356 non-null  str  
dtypes: int64(3), str(3)
memory usage: 3.0 MB


In [70]:
geography_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 43318 entries, 0 to 43317
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   zip         43318 non-null  int64  
 1   city        43318 non-null  str    
 2   latitude    43318 non-null  float64
 3   longitude   43318 non-null  float64
 4   fips        43318 non-null  int64  
 5   county      43318 non-null  str    
 6   population  43318 non-null  int64  
 7   state       43318 non-null  str    
 8   cbsa        43318 non-null  str    
 9   cbsa_name   43318 non-null  str    
dtypes: float64(2), int64(3), str(5)
memory usage: 3.3 MB


In [71]:
quotes_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   EmployerId  7398 non-null   int64
 1   Offered     7398 non-null   str  
 2   Status      7398 non-null   str  
dtypes: int64(1), str(2)
memory usage: 173.5 KB


In [73]:
#Change Zipcode columns to same name in both tables
geography_data.rename(columns={"zip": "ZipCode"}, inplace=True)

# Change merge columns to same Dtypes (numeric)
employer_data["ZipCode"] = pd.to_numeric(
    employer_data["ZipCode"],
    errors="coerce"
).astype("Int64")

# Merge Dataframes
data = (quotes_data.merge(employer_data, on="EmployerId", how="left").merge(geography_data, on="ZipCode", how="left"))
data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7402 entries, 0 to 7401
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   EmployerId       7402 non-null   int64  
 1   Offered          7402 non-null   str    
 2   Status           7402 non-null   str    
 3   ClientTenure     7402 non-null   int64  
 4   Employees        7402 non-null   int64  
 5   Industry         6004 non-null   str    
 6   NetworkStrength  7402 non-null   str    
 7   ZipCode          7395 non-null   Int64  
 8   city             7383 non-null   str    
 9   latitude         7383 non-null   float64
 10  longitude        7383 non-null   float64
 11  fips             7383 non-null   float64
 12  county           7383 non-null   str    
 13  population       7383 non-null   float64
 14  state            7383 non-null   str    
 15  cbsa             7383 non-null   str    
 16  cbsa_name        7383 non-null   str    
dtypes: Int64(1), float64(4), 

### Data Cleaning

In [64]:
# Remove Null values
data_clean = data.dropna()
data_clean.info()

<class 'pandas.DataFrame'>
Index: 5990 entries, 0 to 7401
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   EmployerId       5990 non-null   int64  
 1   Offered          5990 non-null   str    
 2   Status           5990 non-null   str    
 3   ClientTenure     5990 non-null   int64  
 4   Employees        5990 non-null   int64  
 5   Industry         5990 non-null   str    
 6   NetworkStrength  5990 non-null   str    
 7   ZipCode          5990 non-null   Int64  
 8   city             5990 non-null   str    
 9   latitude         5990 non-null   float64
 10  longitude        5990 non-null   float64
 11  fips             5990 non-null   float64
 12  county           5990 non-null   str    
 13  population       5990 non-null   float64
 14  state            5990 non-null   str    
 15  cbsa             5990 non-null   str    
 16  cbsa_name        5990 non-null   str    
dtypes: Int64(1), float64(4), int64

In [ ]:
data_clean = 